In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPool2D, Activation,  Dropout, LSTM, Embedding , Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU, LSTMCell
# from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras.utils import plot_model
from scipy import signal
# from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
# import keras_tuner as kt
# import keras_lmu
from sklearn.model_selection import train_test_split
#import torch
import tensorflow.keras as keras
import tensorflow.keras.backend as keras_backend
import second_gen_RNN
import third_gen_RLIF
import random
from recurrent_LIF import RSNN
#from HSN_QAC import GSN
from Voltage_Neuron import VSN
from Hybrid_Spiking_Neuron import HSNN
from Graded_Spiking_V import GSN
from decoding import DSNN
%matplotlib qt

In [2]:
strain = []
stress = []

training_data   = []
outt   = []
count = 0

for i in range(820):
    


    a   = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Training_data+MAML\input'+str(count+1)+'.csv', delimiter=',')
    b   = np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Training_data+MAML\output'+str(count+1)+'.csv', delimiter=',')
    
    count+=1

    if b.shape[0]>100:
        strain.append(a[0:100,:])
        stress.append(b[0:100].reshape(100,1))

# for i in range(10):
#     training_data.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\d'+str(i+1)+'.csv', delimiter=',')[0:500,:])

# input_NN_scaled = np.array(training_data)[:,:,0:4]
# output_NN_scaled = np.array(training_data)[:,:,4:5]


# for i in range(10):
#     training_data.append(np.genfromtxt(r'C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\d'+str(i+1)+'.csv', delimiter=',')[0:500,:])

# input_NN_scaled = np.array(training_data)[:,:,0:4]
# output_NN_scaled = np.array(training_data)[:,:,4:5]

input_NN   = np.array(strain)
output_NN  = np.array(stress)



input_NN_r            = np.array(strain).reshape(input_NN.shape[0]*input_NN.shape[1], input_NN.shape[2])
output_NN_r           = np.array(stress).reshape(output_NN.shape[0]*output_NN.shape[1], output_NN.shape[2])


scaler1             =   MinMaxScaler(feature_range=(-1, 1))
scaler2             =   MinMaxScaler(feature_range=(0, 1))

input_NN_scaled     =   scaler1.fit_transform(input_NN_r).reshape(input_NN.shape)
output_NN_scaled    =   scaler2.fit_transform(output_NN_r).reshape(output_NN.shape)


input_nn_train_inp, input_nn_validation_inp, input_nn_train_output, input_nn_validation_output = train_test_split(input_NN_scaled, output_NN, test_size=0.85, random_state=38)

input_nn_train_inp, input_nn_test_inp, input_nn_train_output, input_nn_test_output = train_test_split(input_nn_train_inp, input_nn_train_output, test_size=0.5, random_state=38)

In [3]:
def my_loss(y_true,y_pred):
    
    squared_difference = tf.square(y_true - y_pred)
    
    return tf.reduce_mean(squared_difference, axis=[1, 0])

In [4]:
cell1                           = HSNN(208, thr = 1e-10)
cell2                           = HSNN(208, thr = 1e-10)
cell3                           = HSNN(208, thr = 1e-10)
cell4                           = HSNN(208, thr = 1e-10)
cell5                           = HSNN(208, thr = 1e-10)
cell6                           = DSNN(208, thr = 1e-10)




layer1                          = tf.keras.layers.RNN(cell1, return_sequences=True, unroll=True)

layer2                          = tf.keras.layers.RNN(cell2, return_sequences=True, unroll=True)

layer3                          = tf.keras.layers.RNN(cell3, return_sequences=True, unroll=True)

layer4                          = tf.keras.layers.RNN(cell4, return_sequences=True, unroll=True)

layer9                          = tf.keras.layers.RNN(cell5, return_sequences=True, unroll=True)

layer10                          = tf.keras.layers.RNN(cell6, return_sequences=True, unroll=True)

layer5                          = tf.keras.layers.BatchNormalization()

layer6                          = Dense(240, activation='tanh')

layer7                          = Dense(64, activation='tanh')

layer8                          = Dense(1, activation='softplus')

inputs = Input((100, 12))

X     = layer1(inputs)
X2    = layer2(X)
X3    = layer3(X2)
X3    = layer10(X3)

V  = tf.math.tanh(X3)

V  = layer6(V)

V  = layer7(V)

V1  = layer8(V)


model2 = Model(inputs=inputs, outputs=[V1, X, X2, X3])

opt = tf.keras.optimizers.Adam(0.001)

model2.compile(optimizer = opt, loss = my_loss)

model2_ = Model(inputs=inputs, outputs=[V1])


opt = tf.keras.optimizers.legacy.Adam(0.001)

model2_.compile(optimizer = opt, loss = my_loss)

In [5]:
def compute_loss(model, x, y, loss_fn=my_loss):
    
    logits = model(x)
    mse = loss_fn(y, logits)
    
    return mse, logits

In [6]:
# with tf.GradientTape(persistent=True) as train_tape:
    
#     train_loss, _ = compute_loss(layer1, input_nn_train_inp[:,0:20,:], input_nn_train_output[:,0:20,:])
    
#                     # Step 6
#                     #print(f'Train loss: {train_loss}')
# gradients = train_tape.gradient(train_loss, layer1.trainable_variables)

In [7]:
epochs          = 100

In [8]:
input_nn_train  = tf.cast(tf.convert_to_tensor(input_nn_train_inp[:,:,:]), dtype=tf.float32) 
output_nn_train = tf.cast(tf.convert_to_tensor(input_nn_train_output[:,:,:]), dtype=tf.float32)

In [ ]:
#outp, X_, X_2, X_3, X_4 = model2(input_nn_train)

In [9]:
# Prepare the dataset

dataset = tf.data.Dataset.from_tensor_slices((input_nn_train, output_nn_train))
dataset = dataset.shuffle(buffer_size=1024).batch(16)

In [10]:

# Training method

opt = tf.keras.optimizers.Adam(2e-3)
# Custom training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    epoch_loss = 0
    num_batches = 0

    for step, (x_batch, y_batch) in enumerate(dataset):
        with tf.GradientTape() as tape:
            tape.watch(model2_.trainable_variables)
            predictions = model2_(x_batch, training=True)
            loss = my_loss(y_batch, predictions)

        grads = tape.gradient(loss, model2_.trainable_variables)
        opt.apply_gradients(zip(grads, model2_.trainable_variables))

        epoch_loss += loss.numpy()
        num_batches += 1

        if step % 10 == 0:
            print(f"Step {step}: loss = {loss.numpy()}")

    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1} Loss: {avg_loss:}")
    
    if loss<5e-5:
        break


Epoch 1/100
Step 0: loss = [0.40743464]
Epoch 1 Loss: [0.40743464]

Epoch 2/100
Step 0: loss = [0.04129316]
Epoch 2 Loss: [0.04129316]

Epoch 3/100
Step 0: loss = [0.02722274]
Epoch 3 Loss: [0.02722274]

Epoch 4/100
Step 0: loss = [0.02040393]
Epoch 4 Loss: [0.02040393]

Epoch 5/100
Step 0: loss = [0.01626348]
Epoch 5 Loss: [0.01626348]

Epoch 6/100
Step 0: loss = [0.01331053]
Epoch 6 Loss: [0.01331053]

Epoch 7/100
Step 0: loss = [0.01109537]
Epoch 7 Loss: [0.01109537]

Epoch 8/100
Step 0: loss = [0.00940165]
Epoch 8 Loss: [0.00940165]

Epoch 9/100
Step 0: loss = [0.00806426]
Epoch 9 Loss: [0.00806426]

Epoch 10/100
Step 0: loss = [0.00700507]
Epoch 10 Loss: [0.00700507]

Epoch 11/100
Step 0: loss = [0.0061418]
Epoch 11 Loss: [0.0061418]

Epoch 12/100
Step 0: loss = [0.00543762]
Epoch 12 Loss: [0.00543762]

Epoch 13/100
Step 0: loss = [0.00484716]
Epoch 13 Loss: [0.00484716]

Epoch 14/100
Step 0: loss = [0.0043506]
Epoch 14 Loss: [0.0043506]

Epoch 15/100
Step 0: loss = [0.00392407]


In [ ]:
model2_.summary()

In [ ]:
model2_.get_weights()[0].shape

In [ ]:
cell5.decay

In [ ]:
predictions

In [ ]:
y_batch

In [ ]:
a1, a2, a3, a4, a5 = model2(x_batch)

In [ ]:
V_out = model2_(x_batch)

In [ ]:
V_out

In [ ]:
#plt.plot(V[0], label='Spiking based')
plt.plot(y_batch[1], label='Output ')
plt.plot(V_out[1])
#plt.plot(output_nn_train[1], label='Spiking based')
#plt.plot(X_[1][:,207], label='True data')
plt.xlabel('Time')
plt.ylabel('Output')
plt.legend()

In [ ]:
model2(input_nn_train)

In [ ]:
layer4.get_weights()

In [ ]:
output_nn_train

In [ ]:
model2_.fit(input_nn_train_inp[0:1,0:1,:], input_nn_train_output[0:1,0:1,:], epochs=10)

In [ ]:
model2(input_nn_train)

In [ ]:
tf.math.atan(-0.54)

In [ ]:
grads

In [ ]:
grads

In [ ]:
model2_.trainable_variables

In [ ]:
input_nn_train

In [ ]:
V_out

In [ ]:
y_batch

In [ ]:
V_out

In [14]:
weights = model2_.get_weights()

In [16]:
import pickle

path = r'C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Trained_Model\meta_pretrainiertes_model'
with open(f'{path}/klassisch_weights_HSNN.pkl', 'wb') as f:
    pickle.dump(model2_.get_weights() , f)

In [12]:

len(weights)

NameError: name 'weights' is not defined

In [13]:
model2_.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100, 12)]         0         
                                                                 
 rnn (RNN)                   (None, 100, 208)          45760     
                                                                 
 rnn_1 (RNN)                 (None, 100, 208)          86528     
                                                                 
 rnn_2 (RNN)                 (None, 100, 208)          86528     
                                                                 
 rnn_5 (RNN)                 (None, 100, 208)          86528     
                                                                 
 tf.math.tanh (TFOpLambda)   (None, 100, 208)          0         
                                                                 
 dense (Dense)               (None, 100, 240)          5016